《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 분포

이 장에서는 통계학의 가장 기본적인 생각 중 하나인 분포를 소개합니다. 데이터에 어떤 값들이 있고 각 값이 몇 번 나타나는지 정리한 도수분포표부터 시작해, 미국 가족성장조사(NSFG) 자료를 살펴보겠습니다. 이상치라고 부르는 극단적인 값이나 잘못된 값을 찾아보고, 어떻게 다룰지도 생각해 보겠습니다.

[이 노트북의 원문을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/v3/nb/chap02.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from thinkstats import decorate

## 도수분포표

변수를 설명하는 한 가지 방법은 **도수분포표**(frequency table)입니다. 여기에는 변수의 값과 각 값의 **도수**(frequency), 즉 나타난 횟수가 담깁니다. 이처럼 변수의 값들이 어떻게 나타나는지 설명한 것을 그 변수의 **분포**(distribution)라고 합니다.

분포를 표현할 때는 `empiricaldist` 라이브러리를 사용하겠습니다. 여기서 “empirical”은 수학적 모형이 아니라 실제 데이터에 바탕을 둔다는 뜻입니다. `empiricaldist`의 `FreqTab` 클래스로 도수분포표를 계산하고 그릴 수 있습니다. 다음과 같이 가져옵니다.

In [4]:
from empiricaldist import FreqTab

작은 목록으로 사용법을 살펴보겠습니다.

In [5]:
t = [1.0, 2.0, 2.0, 3.0, 5.0]

`FreqTab`의 `from_seq` 메서드는 시퀀스를 받아 `FreqTab` 객체를 만듭니다.

In [6]:
ftab = FreqTab.from_seq(t)
ftab

`FreqTab`은 값과 도수를 담은 pandas `Series`의 한 종류입니다. 이 예에서 `1.0`의 도수는 1, `2.0`의 도수는 2입니다.

`bar` 메서드를 사용하면 도수분포표를 막대그래프로 그릴 수 있습니다.

In [7]:
ftab.bar()
decorate(xlabel="Value", ylabel="Frequency")

`FreqTab`은 pandas `Series`이므로 대괄호 연산자로 어떤 값의 도수를 조회할 수 있습니다.

In [8]:
ftab[2.0]

pandas `Series`와 달리, `FreqTab` 객체를 함수처럼 호출해 값을 조회할 수도 있습니다.

In [9]:
ftab(2.0)

`FreqTab`에 없는 값을 함수 호출 방식으로 조회하면 `0`을 반환합니다.

In [10]:
ftab(4.0)

`FreqTab`의 `qs` 속성은 값들을 배열로 반환합니다. `qs`는 quantities의 줄임말이지만, 엄밀히 말해 모든 값이 수량인 것은 아닙니다.

In [11]:
ftab.qs

`fs` 속성은 도수들을 배열로 반환합니다.

In [12]:
ftab.fs

`items` 메서드로 값과 도수의 쌍을 하나씩 순회할 수 있습니다.

In [13]:
for x, freq in ftab.items():
    print(x, freq)

다른 `FreqTab` 메서드들도 앞으로 살펴보겠습니다.

## NSFG 자료의 분포

새로운 데이터셋을 다룰 때는 사용할 변수를 하나씩 살펴보는 것이 좋습니다. 도수분포표를 확인하는 일은 좋은 출발점입니다.

예로 미국 가족성장조사(NSFG) 자료를 살펴보겠습니다. 앞 장에서는 자료를 내려받아 pandas `DataFrame`으로 읽고, 몇몇 변수를 정제했습니다. 그때 사용한 불러오기와 정제 코드는 `nsfg.py` 모듈에 들어 있습니다. 설치 방법은 이 장의 노트북에 있습니다.

다음 셀들은 데이터 파일을 내려받고, 자료를 읽는 데 필요한 `statadict`를 설치합니다.

In [14]:
try:
    import statadict
except ImportError:
    %pip install statadict

In [15]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/nsfg.py")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

다음과 같이 함수를 가져와 임신 자료 파일을 읽을 수 있습니다.

In [16]:
from nsfg import read_fem_preg

preg = read_fem_preg()

이 장의 예제에서는 생아 출산으로 끝난 임신을 살펴보겠습니다. `query` 메서드로 `outcome`이 1인 행을 선택합니다.

In [17]:
live = preg.query("outcome == 1")

`query`에 전달하는 문자열에서 `outcome` 같은 변수 이름은 `DataFrame`의 열 이름을 가리킵니다. 문자열 안에는 `==` 같은 연산자와 `1` 같은 피연산자도 쓸 수 있습니다.

이제 `FreqTab.from_seq`로 `birthwgt_lb`의 각 값이 나타나는 횟수를 셀 수 있습니다. 이 열에는 출생 체중의 파운드 부분이 들어 있습니다. `name` 인수는 `FreqTab` 객체에 이름을 붙이며, 그래프를 그릴 때 이 이름이 범례에 쓰입니다.

In [18]:
ftab_lb = FreqTab.from_seq(live["birthwgt_lb"], name="birthwgt_lb")

분포를 그리면 다음과 같습니다.

In [19]:
ftab_lb.bar()
decorate(xlabel="Pounds", ylabel="Frequency")

이런 분포를 보면 먼저 모양이 눈에 들어옵니다. 흔히 말하는 종 모양 곡선과 비슷합니다. 이런 모양을 나타내는 대표적인 모형이 정규분포, 또는 가우스분포입니다. 또 눈여겨볼 것은 가장 자주 나타나는 값인 **최빈값**(mode)입니다. `idxmax` 메서드는 도수가 가장 큰 값에 대응하는 인덱스를 찾아주므로, 최빈값을 구하는 데 쓸 수 있습니다.

In [20]:
ftab_lb.idxmax()

`FreqTab`의 `mode` 메서드도 같은 일을 합니다.

In [21]:
ftab_lb.mode()

이 분포의 최빈값은 7파운드입니다.

다른 예로, 출생 체중의 온스 부분인 `birthwgt_oz`의 도수분포표를 보겠습니다.

In [22]:
ftab_oz = FreqTab.from_seq(live["birthwgt_oz"], name="birthwgt_oz")
ftab_oz.bar()
decorate(xlabel="Ounces", ylabel="Frequency")

파운드와 온스는 사람이 정한 단위이므로, 온스 부분의 값들은 대략 비슷한 확률로 나타날 것으로 예상할 수 있습니다. 즉 **균등분포**(uniform distribution)를 기대할 수 있습니다. 하지만 `0`은 다른 값보다 자주 나타나고, `1`과 `15`는 덜 나타납니다. 출생 체중이 정수 파운드에 가까우면 응답자가 반올림해 답하는 듯합니다.

이번에는 임신이 끝났을 때 어머니의 나이인 `agepreg`의 도수분포표를 보겠습니다.

In [23]:
ftab_age = FreqTab.from_seq(live["agepreg"], name="agepreg")

NSFG에서 나이는 연과 월 단위로 기록되므로, 앞서 본 분포들보다 서로 다른 값이 많습니다. 막대들이 너무 많이 겹치지 않도록 `bar` 메서드에 키워드 인수 `width=0.1`을 전달해 너비를 조절하겠습니다.

In [24]:
ftab_age.bar(width=0.1)
decorate(xlabel="Age", ylabel="Frequency")

대체로 종 모양이지만 오른쪽으로 **치우쳐**(skewed) 있습니다. 꼬리가 왼쪽보다 오른쪽으로 더 길게 뻗어 있다는 뜻입니다.

마지막으로 주 단위 임신 기간인 `prglngth`의 도수분포표를 보겠습니다. `xlim` 인수로 x축 범위를 20주에서 50주로 정합니다. 이 범위 밖의 값은 많지 않고, 아마 오류일 것입니다.

In [25]:
ftab_length = FreqTab.from_seq(live["prglngth"], name="prglngth")
ftab_length.bar()
decorate(xlabel="Weeks", ylabel="Frequency", xlim=[20, 50])

가장 흔한 값은 단연 39주입니다. 왼쪽 꼬리가 오른쪽보다 깁니다. 일찍 태어나는 아기는 흔하지만 임신이 43주를 넘기는 경우는 드물고, 그 정도가 되면 의사가 개입하는 일이 많기 때문입니다.

## 이상치

도수분포표에서는 분포의 모양과 자주 나타나는 값을 쉽게 알아볼 수 있지만, 드문 값은 잘 보이지 않을 수 있습니다. 다음으로 넘어가기 전에 **이상치**(outlier)를 확인하는 것이 좋습니다. 이상치는 극단적인 값입니다. 측정이나 기록 오류일 수도 있고, 드물지만 실제로 일어난 일을 정확히 기록한 것일 수도 있습니다.

다음 함수는 `FreqTab` 객체와 정수 `n`을 받아, 슬라이스 인덱스로 가장 작은 값 `n`개와 그 도수를 선택합니다.

In [26]:
def smallest(ftab, n=10):
    return ftab[:n]

`prglngth`의 도수분포표에서 가장 작은 값 10개는 다음과 같습니다.

In [27]:
smallest(ftab_length)

생아 출산에 해당하는 행만 골랐으므로, 임신 기간이 10주 미만인 값은 분명 오류입니다. 가장 그럴듯한 설명은 임신 결과가 잘못 부호화되었다는 것입니다. 30주보다 긴 값은 대체로 타당해 보입니다. 10~30주 사이는 확신하기 어렵습니다. 일부는 오류겠지만, 일부는 조산을 정확히 기록한 값일 것입니다.

다음 함수는 `FreqTab`에서 가장 큰 값들을 선택합니다.

In [28]:
def largest(ftab, n=10):
    return ftab[-n:]

자료에서 가장 긴 임신 기간은 다음과 같습니다.

In [29]:
largest(ftab_length)

여기에도 오류로 보이는 값이 있습니다. 원문에서는 대부분의 의사가 임신 41주를 넘기면 유도분만을 권한다고 설명하며, 따라서 50주라는 기록은 맞을 가능성이 낮다고 봅니다. 하지만 분명한 오류와, 드물게 일어난 일을 정확히 기록한 값 사이에 선을 긋기는 어렵습니다.

이상치를 어떻게 다루는 것이 좋은지는 “분야 지식”, 즉 데이터가 어디에서 왔고 무엇을 뜻하는지에 관한 지식에 달려 있습니다. 어떤 분석을 하려는지에 따라서도 달라집니다.

이 예제에서 처음 던진 질문은 첫아이가 다른 아이들보다 일찍 태어나는지, 늦게 태어나는지였습니다. 따라서 잘못된 값이 조금 섞여 있어도 크게 흔들리지 않는 통계량을 사용하겠습니다.

## 첫아이

이제 첫아이와 그 밖의 아이들의 임신 기간 분포를 비교하겠습니다. `query` 메서드로 첫아이에 해당하는 행과 나머지 행을 선택합니다.

In [30]:
firsts = live.query("birthord == 1")
others = live.query("birthord != 1")

각 집단의 임신 기간으로 `FreqTab`을 만듭니다.

In [31]:
ftab_first = FreqTab.from_seq(firsts["prglngth"], name="firsts")
ftab_other = FreqTab.from_seq(others["prglngth"], name="others")

다음 함수는 두 도수분포표의 막대를 나란히 그립니다.

In [32]:
def two_bar_plots(ftab1, ftab2, width=0.45):
    ftab1.bar(align="edge", width=-width)
    ftab2.bar(align="edge", width=width, alpha=0.5)

결과는 다음과 같습니다.

In [33]:
two_bar_plots(ftab_first, ftab_other)
decorate(xlabel="Weeks", ylabel="Frequency", xlim=[20, 50])

분포의 모양이나 이상치에서 뚜렷한 차이는 보이지 않습니다. 첫아이가 아닌 경우 39주에 태어나는 수가 더 많아 보입니다. 하지만 자료에 첫아이가 아닌 아이가 더 많이 들어 있으므로, 단순한 건수를 곧바로 비교해서는 안 됩니다.

In [34]:
firsts["prglngth"].count(), others["prglngth"].count()

분포의 평균을 비교하면 첫아이가 평균적으로 조금 더 늦게 태어나는 듯합니다.

In [35]:
first_mean = firsts["prglngth"].mean()
other_mean = others["prglngth"].mean()
first_mean, other_mean

하지만 차이는 0.078주에 불과합니다. 약 13시간입니다.

In [36]:
diff = first_mean - other_mean
diff, diff * 7 * 24

겉으로 드러난 이 차이에는 몇 가지 설명이 가능합니다.

- 첫아이와 다른 아이 사이에 평균 임신 기간의 실제 차이가 있을 수 있습니다.
- 표본을 뽑는 과정, 즉 조사 응답자를 선택하는 과정의 편향 때문에 이런 차이가 나타났을 수 있습니다.
- 측정 오류 때문일 수 있습니다. 예를 들어 응답자가 기억해 보고한 임신 기간의 정확도가 두 집단에서 다를 수 있습니다.
- 표본을 뽑는 과정에서 생긴 우연한 변동의 결과일 수 있습니다.

뒤의 장에서 이 설명들을 더 자세히 검토하겠습니다. 지금 확인할 수 있는 것은 이 표본에서 두 집단의 평균 임신 기간에 작은 차이가 있다는 점입니다.

## 효과 크기

이런 차이를 “효과”라고 부르기도 합니다. 효과가 얼마나 큰지 수치로 나타내는 방법은 여러 가지입니다. 가장 간단한 방법은 절대적인 차이를 보고하는 것입니다. 이 예에서는 0.078주입니다.

상대적인 차이를 보고할 수도 있습니다. 예를 들어 첫아이의 임신 기간이 나머지보다 평균 0.2% 길다고 말할 수 있습니다.

In [37]:
diff / live["prglngth"].mean() * 100

또 다른 방법은 **표준화한**(standardized) 효과 크기를 보고하는 것입니다. 단위나 집단이 달라도 차이의 크기를 비교할 수 있도록 만든 통계량입니다.

표준화한다는 것은 차이를 표준편차의 몇 배인지로 표현한다는 뜻입니다. 그래서 다음과 같이 계산하고 싶을 수 있습니다.

In [38]:
diff / live["prglngth"].std()

하지만 여기서는 두 집단을 합쳐 표준편차를 계산했습니다. 두 집단의 평균이 다르면, 합친 자료의 표준편차에는 집단 사이의 차이도 반영됩니다. 이를 분모로 쓰면 효과 크기가 작아 보일 수 있습니다.

한 집단의 표준편차만 사용하는 방법도 있지만, 어느 쪽을 골라야 할지 분명하지 않습니다. 두 표준편차를 단순히 평균할 수도 있겠지만, 집단 크기가 다르면 한 집단에는 지나치게 큰 가중치를, 다른 집단에는 지나치게 작은 가중치를 주게 됩니다.

흔히 사용하는 해결책은 **통합 표준편차**(pooled standard deviation)입니다. 각 집단의 분산을 가중평균한 통합 분산에 제곱근을 취한 값입니다. 먼저 각 집단의 분산을 구하겠습니다.

In [39]:
group1, group2 = firsts["prglngth"], others["prglngth"]

In [40]:
v1, v2 = group1.var(), group2.var()

집단 크기를 가중치로 사용해 다음과 같이 가중평균합니다.

In [41]:
n1, n2 = group1.count(), group2.count()
pooled_var = (n1 * v1 + n2 * v2) / (n1 + n2)

마지막으로 제곱근을 취하면 통합 표준편차가 됩니다.

In [42]:
np.sqrt(pooled_var)

통합 표준편차는 각 집단의 표준편차 사이에 있습니다.

In [43]:
firsts["prglngth"].std(), others["prglngth"].std()

통합 표준편차를 이용한 표준화 효과 크기를 **코언의 효과 크기**(Cohen's effect size)라고 합니다. 다음 함수로 계산할 수 있습니다.

In [44]:
def cohen_effect_size(group1, group2):
    diff = group1.mean() - group2.mean()

    v1, v2 = group1.var(), group2.var()
    n1, n2 = group1.count(), group2.count()
    pooled_var = (n1 * v1 + n2 * v2) / (n1 + n2)

    return diff / np.sqrt(pooled_var)

평균 임신 기간 차이의 효과 크기는 다음과 같습니다.

In [45]:
cohen_effect_size(firsts["prglngth"], others["prglngth"])

이 예에서 차이는 표준편차의 0.029배로 작습니다. 비교하자면 남녀의 키 차이는 표준편차의 약 1.7배입니다.

## 결과 보고하기

첫아이와 다른 아이 사이에 임신 기간 차이가 있다면, 그 차이를 설명하는 몇 가지 방법을 살펴보았습니다. 결과를 어떻게 보고해야 할까요?

누가 질문하는지에 따라 답이 달라집니다. 과학자는 아무리 작더라도 실제로 존재하는 효과라면 관심이 있을 수 있습니다. 의사는 **실질적으로 의미 있는**(practically significant) 효과, 즉 실제 진료에서 중요할 만큼 큰 차이에만 관심이 있을 수 있습니다. 임신한 사람은 일찍 또는 늦게 출산할 확률처럼 자신과 관련 있는 결과를 알고 싶을 것입니다.

결과를 보고하는 방식은 목적에 따라서도 달라집니다. 효과가 중요하다는 점을 보여주려면 차이가 두드러지는 요약 통계량을 고를 수 있습니다. 환자를 안심시키려면 차이가 전체 맥락에서 어느 정도인지 보여 주는 통계량을 고를 수 있습니다.

보고서와 시각화는 결과가 분명히 드러나도록 구성해야 합니다. 다만 설득하려는 목적 때문에 수치를 왜곡해서는 안 됩니다. 불확실성과 한계도 함께 밝혀야 합니다.

## 용어 정리

- **분포**(distribution): 데이터셋에 있는 값들과 각 값이 나타나는 빈도.
- **도수분포표**(frequency table): 각 값을 도수에 대응시킨 표.
- **도수**(frequency): 표본에서 어떤 값이 나타난 횟수.
- **치우친 분포**(skewed): 비대칭이며 한쪽의 극단적인 값들이 반대쪽보다 더 멀리 뻗어 있는 분포.
- **최빈값**(mode): 표본에서 가장 자주 나타나는 값. 여러 개일 수도 있습니다.
- **균등분포**(uniform distribution): 가능한 값들의 확률이 같은 분포.
- **이상치**(outlier): 분포에서 극단적으로 크거나 작은 값.
- **표준화**(standardized): 서로 다른 데이터셋이나 분야에서도 비교할 수 있는 방식으로 통계량을 표현하는 것.
- **통합 표준편차**(pooled standard deviation): 둘 이상의 집단에서 얻은 정보를 결합해 공통 표준편차를 계산한 통계량.
- **코언의 효과 크기**(Cohen's effect size): 두 집단의 평균 차이를 수치로 나타내는 표준화 통계량.
- **실질적으로 의미 있는 효과**(practically significant): 실제로 중요할 만큼 큰 효과.

## 연습문제

이 장의 연습문제에서는 여성 응답자 한 명당 한 행이 있는 NSFG 여성 응답자 파일을 불러오겠습니다. 자료와 코드북을 내려받는 방법은 이 장의 노트북에 있습니다.

In [46]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemResp.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemResp.dat.gz")

여성 응답자 파일의 [코드북](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NSFG/Cycle6Codebook-Female.pdf)을 참고하세요.

`nsfg.py` 모듈에는 여성 응답자 파일을 읽고 일부 변수를 정제한 다음 `DataFrame`을 반환하는 함수가 있습니다.

In [47]:
from nsfg import read_fem_resp

resp = read_fem_resp()
resp.shape

이 `DataFrame`에는 열이 3,092개 있지만, 여기서는 일부만 사용하겠습니다.

### 연습문제 2.1

응답자 가족의 총소득을 1~14의 값으로 기록한 `totincr`부터 살펴보겠습니다. 각 값이 어떤 소득 수준을 뜻하는지는 응답자 파일의 코드북에서 확인할 수 있습니다.

이 변수의 분포를 나타내는 `FreqTab` 객체를 만들고 막대그래프로 그려 보세요.

In [48]:
# Solution goes here

### 연습문제 2.2

응답자가 낳은 자녀 수를 기록한 `parity` 열의 도수분포표를 만드세요. 이 분포의 모양을 어떻게 설명할 수 있나요?

In [49]:
# Solution goes here

In [50]:
# Solution goes here

`largest` 함수로 `parity`의 가장 큰 값들을 찾아보세요. 오류라고 생각되는 값이 있나요?

In [51]:
# Solution goes here

### 연습문제 2.3

소득이 높은 여성과 낮은 여성 중 어느 쪽이 자녀를 더 많이 낳는지 살펴보겠습니다. `query` 메서드로 소득이 가장 높은 수준인 14에 해당하는 응답자를 선택하세요. 이 응답자들만 대상으로 `parity`의 도수분포표를 그리세요.

In [52]:
# Solution goes here

소득이 가장 높은 응답자들과 나머지 응답자들의 평균 `parity`를 비교하세요.

In [53]:
# Solution goes here

이 차이에 대한 코언의 효과 크기를 계산하세요. 첫아이와 다른 아이 사이의 임신 기간 차이에 비하면 어떤가요?

In [54]:
# Solution goes here

이 결과가 소득이 높은 사람은 자녀를 적게 낳는다는 것을 보여주나요? 아니면 관측된 차이를 설명할 다른 이유를 생각할 수 있나요?

In [55]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)